In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import StaleElementReferenceException, ElementClickInterceptedException, NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import os
import glob
import time
import random
import csv

In [2]:
# Ścieżki
OUTPUT_FOLDER = r"D:\Base work dir\ING project\Pre-selection assignment"
CSV_VISITED_PATH = os.path.join(OUTPUT_FOLDER, "visited_links.csv")


# Adres strony głównej
BASE_URL = "https://www.otodom.pl/pl/wyniki/sprzedaz/kawalerka/cala-polska"

# Zamknięcie pop-upu cookies
def close_cookies_popup(driver):
    try:
        cookie_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button#onetrust-accept-btn-handler"))
        )
        cookie_button.click()
        print("Zamknięto okno cookies.")
        time.sleep(2)
    except TimeoutException:
        print("Brak okna cookies do zamknięcia.")

def expand_building_details(driver):
    try:

        header = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//header[p[contains(text(), "Budynek i materiały")]]'))
        )

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", header)
        time.sleep(1)
        header.click()
        time.sleep(2)  

        details_section = driver.find_elements(By.XPATH, '//div[contains(@class, "css-1ftuvmu") and not(@hidden)]')
        if details_section:
            print("Sekcja 'Budynek i materiały' rozwinięta!")
        else:
            print("Selenium nadal nie widzi sekcji!")

    except Exception as e:
        print(f"Błąd: {e}")

def get_value_after_label(label, driver):
    try:
        element = driver.find_element(By.XPATH, f'//p[text()="{label}"]/following-sibling::p')
        return element.text.strip()
    except NoSuchElementException:
        return "Brak danych"

def get_value_from_details(label, driver):
    try:
        time.sleep(2) 

        elements = driver.find_elements(By.XPATH, f'//p[contains(text(), "{label}")]/following-sibling::p')
        if elements:
            return elements[0].text.strip()
        else:
            print(f"Selenium nie znalazło '{label}'")
            return "Brak danych"
    except Exception as e:
        print(f"Błąd przy pobieraniu '{label}': {e}")
        return "Brak danych"


def get_value_from_buttons(index, driver):
    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, 'button.eezlw8k1.css-1nk40gi')
        if len(buttons) > index:
            return buttons[index].find_element(By.CSS_SELECTOR, 'div.css-1ftqasz').text.strip()
        return "Brak danych"
    except NoSuchElementException:
        return "Brak danych"

# Pobieranie linków do ofert
def get_offer_links(driver, max_links=1000):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)  

    close_cookies_popup(driver)

    all_links = set()
    previous_links = set() 

    while len(all_links) < max_links:
        try:
            # Pobranie ofert
            offers = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'a[href^="/pl/oferta/"]')))
            new_links = {offer.get_attribute('href') for offer in offers if offer.get_attribute('href')}
            
            # Sprawdzenie, czy strona rzeczywiście się zmieniła
            if new_links == previous_links:
                print("Strona nie zmieniła ofert. Próbuję ponownie...")
                time.sleep(3)
                continue  # Powtórz próbę pobrania

            previous_links = new_links  # Aktualizacja poprzednich linków
            all_links.update(new_links)
            print(f"🔹 Pobrano {len(new_links)} nowych ofert, łącznie: {len(all_links)}")

            # Sprawdzenie limitu ofert
            if len(all_links) >= max_links:
                print("Osiągnięto limit ofert, kończenie zbierania linków.")
                break

            # Przewinięcie do przycisku "Następna strona"
            try:
                next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//li[@title="Go to next Page" and @aria-disabled="false"]')))
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
                time.sleep(random.uniform(1, 2))

                # Ponawianie prób kliknięcia
                attempts = 0
                while attempts < 3:
                    try:
                        next_button.click()
                        time.sleep(random.uniform(4, 6))  # Dłuższe czekanie na załadowanie
                        break
                    except ElementClickInterceptedException:
                        print("Kliknięcie nieudane, ponawiam próbę...")
                        driver.execute_script("arguments[0].click();", next_button)
                        time.sleep(1)
                        attempts += 1

            except (NoSuchElementException, TimeoutException):
                print("Nie znaleziono przycisku paginacji. Koniec zbierania linków.")
                break

        except StaleElementReferenceException:
            print("Strona została zaktualizowana, ponawiam próbę...")
            continue  

    return list(all_links)[:max_links]   

# Pobieranie szczegółów oferty
def get_offer_details(offer_url, driver):
    driver.get(offer_url)
    time.sleep(random.uniform(2, 4))

    data = {"URL": offer_url}

    def get_text(selector, driver):
        try:
            return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
        except:
            return "Brak danych"

    data["Cena"] = get_text('strong[data-cy="adPageHeaderPrice"]', driver)
    data["Cena za m²"] = get_text('div[aria-label="Cena za metr kwadratowy"]', driver)
    data["Adres"] = get_text('a.css-1jjm9oe', driver)
    data["Powierzchnia (m²)"] = get_value_from_buttons(0, driver)
    data["Liczba pokoi"] = get_value_from_buttons(1, driver)
    data["Ogrzewanie"] = get_value_after_label("Ogrzewanie", driver)
    data["Rynek"] = get_value_after_label("Rynek", driver)
    data["Typ ogłoszeniodawcy"] = get_value_after_label("Typ ogłoszeniodawcy", driver)

    expand_building_details(driver)
    time.sleep(8)
    data["Rok budowy"] = get_value_from_details("Rok budowy", driver)
    time.sleep(4)
    data["Rodzaj zabudowy"] = get_value_from_details("Rodzaj zabudowy", driver)
    time.sleep(4)
    data["Okna"] = get_value_from_details("Okna", driver)
    time.sleep(4)
    data["Materiał budynku"] = get_value_from_details("Materiał budynku", driver)

    print("\n Oferta pobrana:")
    for key, value in data.items():
        print(f"   {key}: {value}")

    return data


def save_chunk_to_csv(data_list, chunk_index):
    """
    Zapisuje listę słowników `data_list` do pliku:
    data_studio_pt{chunk_index}.csv

    w formacie CSV (DictWriter).
    """
    if not data_list:
        return  # Nie ma czego zapisywać

    file_name = f"data_studio_pt{chunk_index}.csv"
    full_path = os.path.join(OUTPUT_FOLDER, file_name)

    # Otwieramy w trybie 'w' (nadpisujemy), bo zakładamy jeden plik na każdą paczkę
    with open(full_path, mode='w', newline='', encoding='utf-8') as f:
        fieldnames = data_list[0].keys()  # Zakładamy, że wszystkie słowniki mają te same klucze
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data_list)

    print(f"  ➜ Zapisano {len(data_list)} ofert do pliku: {file_name}")


def scrape_otodom(offer_links, driver, chunk_size=2):
    """
    - Wczytuje listę "odwiedzonych" linków z pliku visited_links.csv (jeśli istnieje).
    - Dla każdego linku, którego NIE ma w visited_links, pobiera dane.
    - Co `chunk_size` ofert zapisuje je do nowego pliku data_studio_ptX.csv.
    - Każdy nowo przetworzony link trafia natychmiast do visited_links.csv, aby nie był
      ponownie przetwarzany po ewentualnym restarcie skryptu.
    """

    # 1) Wczytaj dotychczas odwiedzone linki
    visited_links = set()
    if os.path.exists(CSV_VISITED_PATH):
        with open(CSV_VISITED_PATH, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                visited_links.add(row[0])  # Wpisujemy link do zbioru

    print(f"\nOdczytano {len(visited_links)} linków z visited_links.csv")

    # Tutaj gromadzimy dane do jednej "paczkowej" porcji
    all_data = []
    chunk_index = 1

    for idx, link in enumerate(offer_links, start=1):
        # 2) Sprawdź czy link już był przetworzony
        if link in visited_links:
            print(f"[{idx}] Link już przetworzony: {link}")
            continue

        # 3) Jeśli nie, pobierz dane oferty
        print(f"[{idx}] Pobieranie oferty: {link}")
        details = get_offer_details(link, driver)
        all_data.append(details)

        # 4) Dodaj go do visited_links "natychmiast", by uniknąć wielokrotnego przetwarzania
        visited_links.add(link)
        # Dopisujemy go też do pliku visited_links.csv w trybie 'append'
        with open(CSV_VISITED_PATH, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([link])

        # 5) Czy mamy już pełną "paczkę"? Jeśli tak – zapisz ją i wyczyść
        if len(all_data) >= chunk_size:
            save_chunk_to_csv(all_data, chunk_index)
            all_data = []
            chunk_index += 1


    # 6) Po zakończeniu pętli zapisz pozostałe oferty (jeżeli coś zostało)
    if all_data:
        save_chunk_to_csv(all_data, chunk_index)


In [3]:
# Konfiguracja Selenium
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Uruchamianie WebDrivera
service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)

In [ ]:
offer_links = get_offer_links(driver, 2000)

In [ ]:
scrape_otodom(offer_links, driver, 50)

In [52]:
driver.quit() 

In [8]:
# Wzorzec wyszukiwania plików
pattern = os.path.join(OUTPUT_FOLDER, "data_studio_pt*.csv")

# Nazwa finalnego pliku, do którego zepniemy wszystkie CSV-y
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "data_studio_merged.csv")

def merge_data_studio_csv():
    # Znajdź wszystkie pliki w folderze pasujące do wzorca
    files = glob.glob(pattern)
    # Sortujemy listę, by pliki wczytywać w kolejności pt1, pt2, pt3...
    files.sort()

    all_data = []
    header = None

    # Wczytujemy każdy plik po kolei
    for file_path in files:
        print(f"  ➜ Łączę plik: {os.path.basename(file_path)}")

        with open(file_path, mode="r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            # Ustawiamy header przy pierwszym pliku
            if header is None:
                header = reader.fieldnames

            # Dodajemy każdą linijkę do naszej wspólnej listy
            for row in reader:
                all_data.append(row)

    # Teraz zapisujemy wszystko do jednego pliku OUTPUT_FILE
    if not all_data:
        print("Brak danych do łączenia!")
        return

    print(f"\n  ➜ Zapisuję {len(all_data)} rekordów do pliku: {os.path.basename(OUTPUT_FILE)}")
    with open(OUTPUT_FILE, mode="w", newline="", encoding="utf-8") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=header)
        writer.writeheader()
        writer.writerows(all_data)

    print("Zakończono łączenie plików.")

merge_data_studio_csv()


  ➜ Łączę plik: data_studio_pt1.csv
  ➜ Łączę plik: data_studio_pt10.csv
  ➜ Łączę plik: data_studio_pt11.csv
  ➜ Łączę plik: data_studio_pt12.csv
  ➜ Łączę plik: data_studio_pt13.csv
  ➜ Łączę plik: data_studio_pt14.csv
  ➜ Łączę plik: data_studio_pt15.csv
  ➜ Łączę plik: data_studio_pt16.csv
  ➜ Łączę plik: data_studio_pt17.csv
  ➜ Łączę plik: data_studio_pt18.csv
  ➜ Łączę plik: data_studio_pt19.csv
  ➜ Łączę plik: data_studio_pt2.csv
  ➜ Łączę plik: data_studio_pt20.csv
  ➜ Łączę plik: data_studio_pt21.csv
  ➜ Łączę plik: data_studio_pt22.csv
  ➜ Łączę plik: data_studio_pt23.csv
  ➜ Łączę plik: data_studio_pt24.csv
  ➜ Łączę plik: data_studio_pt3.csv
  ➜ Łączę plik: data_studio_pt4.csv
  ➜ Łączę plik: data_studio_pt5.csv
  ➜ Łączę plik: data_studio_pt6.csv
  ➜ Łączę plik: data_studio_pt7.csv
  ➜ Łączę plik: data_studio_pt8.csv
  ➜ Łączę plik: data_studio_pt9.csv

  ➜ Zapisuję 1200 rekordów do pliku: data_studio_merged.csv
Zakończono łączenie plików.
